In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import os

print('Libraries loaded')

Libraries loaded


In [4]:
# Load raw data
data_path = '../data/raw/'

files = [
    'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',
    'Wednesday-workingHours.pcap_ISCX.csv',
]

dfs = []
for f in files:
    path = os.path.join(data_path, f)
    if os.path.exists(path):
        temp = pd.read_csv(path, low_memory=False)
        dfs.append(temp)
        print(f'Loaded {f}')

df = pd.concat(dfs, ignore_index=True)
df.columns = df.columns.str.strip()
print(f'Combined shape: {df.shape}')

Loaded Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Loaded Wednesday-workingHours.pcap_ISCX.csv
Combined shape: (918448, 79)


In [5]:
# Step 1 — Fix infinite and NaN values
df.replace([np.inf, -np.inf], np.nan, inplace=True)
before = len(df)
df.dropna(inplace=True)
after = len(df)
print(f'Rows removed: {before - after:,}')
print(f'Remaining rows: {after:,}')

Rows removed: 1,331
Remaining rows: 917,117


In [6]:
# Step 2 — Binary label encoding
# 0 = normal, 1 = attack
df['Label'] = df['Label'].apply(lambda x: 0 if x.strip() == 'BENIGN' else 1)
print('Label encoding done')
print(df['Label'].value_counts())

Label encoding done
Label
0    537369
1    379748
Name: count, dtype: int64


In [7]:
# Step 3 — Select features
selected_features = [
    'Destination Port', 'Flow Duration', 'Total Fwd Packets',
    'Total Backward Packets', 'Total Length of Fwd Packets',
    'Total Length of Bwd Packets', 'Fwd Packet Length Max',
    'Fwd Packet Length Min', 'Fwd Packet Length Mean',
    'Bwd Packet Length Max', 'Bwd Packet Length Min',
    'Bwd Packet Length Mean', 'Flow Bytes/s', 'Flow Packets/s',
    'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
    'Fwd IAT Total', 'Bwd IAT Total', 'Fwd PSH Flags',
    'Fwd Header Length', 'Bwd Header Length',
    'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length',
    'Max Packet Length', 'Packet Length Mean', 'Packet Length Std',
    'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count',
    'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count',
    'URG Flag Count', 'Average Packet Size', 'Avg Fwd Segment Size',
    'Avg Bwd Segment Size'
]

available = [f for f in selected_features if f in df.columns]
missing_f = [f for f in selected_features if f not in df.columns]

print(f'Features available: {len(available)}')
print(f'Features missing:   {len(missing_f)}')
if missing_f:
    print(f'Missing: {missing_f}')

X = df[available]
y = df['Label']
print(f'\nX shape: {X.shape}')
print(f'y shape: {y.shape}')

Features available: 39
Features missing:   0

X shape: (917117, 39)
y shape: (917117,)


In [8]:
# Step 4 — Train/test split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Training set: {X_train.shape[0]:,} rows')
print(f'Test set:     {X_test.shape[0]:,} rows')

Training set: 733,693 rows
Test set:     183,424 rows


In [9]:
# Step 5 — Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print('Scaling done')

Scaling done


In [10]:
# Step 6 — Save everything
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

np.save('../data/processed/X_train.npy', X_train_scaled)
np.save('../data/processed/X_test.npy',  X_test_scaled)
np.save('../data/processed/y_train.npy', y_train.values)
np.save('../data/processed/y_test.npy',  y_test.values)
joblib.dump(scaler, '../models/scaler.pkl')

# Save feature names
pd.Series(available).to_csv('../data/processed/feature_names.csv', index=False)

print('All files saved')
print(f'X_train: {X_train_scaled.shape}')
print(f'X_test:  {X_test_scaled.shape}')

All files saved
X_train: (733693, 39)
X_test:  (183424, 39)
